[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/093.%20The%20Inventory-Routing%20Problem%20%28IRP%29/P93-Tier-7_executed.ipynb)

---


# 93. The Inventory-Routing Problem (IRP)
## Tier 7: The Human-AI Symbiotic Partnership

### Purpose and scope

This notebook illustrates a human-in-the-loop decision-support workflow for an inventory-routing setting. An AI component produces explainable route and inventory recommendations; a simulated domain expert reviews them; feedback is recorded; and simple trust and collaboration indicators are updated over a planning horizon.

This is a conceptual partnership simulation, not an optimization model that solves the full inventory-routing problem. The nearest-neighbor route builder is a simple heuristic used to make route recommendations; it does not guarantee a shortest route. Inventory levels and demand are held fixed during the periods, and recommendations are not executed against a vehicle-loading or inventory-balance model. The simulation therefore measures the behavior of the stated feedback rules, not operational cost savings, service improvements, calibrated confidence, or real-world trust.

### Inputs and outputs

The example has one depot, 15 geographically distributed customers, four vehicles, and an eight-period planning horizon. Customer locations, demand parameters, and starting inventories are generated with NumPy's seed of 42. The system reports recommendations, simulated expert feedback, and period-level collaboration indicators. For demonstration, it generates route recommendations for at most two vehicles and inventory recommendations for at most three urgent customers per period.

### Modeling assumptions

- The depot is located at the origin, and distance is Euclidean distance in the coordinate units used by the generated instance.
- A customer's days of stock are estimated as current inventory divided by mean daily demand.
- Urgency thresholds, confidence rules, risk scores, and the simulated expert's review criteria are illustrative rules, not empirically calibrated quantities.
- The inventory snapshot is not updated between periods. Feedback is recorded, but expert patterns are not used to retrain a predictive model.
- Trust indicators are descriptive summaries of the simulation's rules. In particular, agreement is not evidence that a recommendation is operationally correct.

### Collaborative decision process

The teaching sequence is: (1) generate explainable AI recommendations, (2) have the simulated expert accept or modify each recommendation, (3) record feedback, and (4) update trust and performance indicators. A human reviewer can add contextual knowledge that is absent from a mathematical model; here, that contribution is represented by transparent, deterministic rules rather than a real human interface.

### Interpreting the results

Use the reported agreement, confidence, and decision-quality values to understand how the implemented rules interact. They are not independent measurements of decision accuracy. The comparison section uses stated illustrative reference scores for other approaches; those scores are not generated by experiments in this notebook. A high score or agreement rate should therefore be interpreted as behavior within this example, not as proof that partnership is superior in general.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class AIRecommendation:
    """A recommendation with a heuristic confidence score and explanation."""

    recommendation_type: str
    action: str
    confidence: float
    reasoning: List[str]
    alternatives: List[Dict[str, Any]]
    expected_impact: Dict[str, float]
    risk_assessment: Dict[str, float]


@dataclass
class HumanFeedback:
    """Simulated expert feedback on an AI recommendation."""

    recommendation_id: str
    accepted: bool
    modifications: Dict[str, Any]
    reasoning: str
    confidence_in_decision: float
    expert_notes: str


@dataclass
class TrustMetrics:
    """Descriptive partnership indicators, each on a 0-to-1 scale."""

    overall_trust: float
    prediction_accuracy: float
    reliability_score: float
    agreement_rate: float
    learning_progress: float
    relationship_strength: float


class ExplainableAI:
    """Generate transparent, rule-based recommendations for the example."""

    def __init__(self, depot, customers, vehicles, num_periods):
        self.depot = depot
        self.customers = customers
        self.vehicles = vehicles
        self.num_periods = num_periods
        self.expert_patterns = []
        self.decision_history = []
        self.performance_history = []
        self.current_inventory = {}
        self.current_routes = {}
        self.current_costs = {}
        self.rng = np.random.default_rng(42)
        self._initialize_state()

    def _initialize_state(self):
        """Create the inventory and route snapshots used by the rules."""
        for customer in self.customers:
            self.current_inventory[customer.id] = customer.initial_inventory
        for vehicle in self.vehicles:
            self.current_routes[vehicle.id] = []
            self.current_costs[vehicle.id] = 0.0

    def generate_route_recommendation(self, period: int, vehicle_id: int) -> AIRecommendation:
        """Build a heuristic route recommendation and explain its inputs."""
        vehicle = next(v for v in self.vehicles if v.id == vehicle_id)
        urgent_customers = self._identify_urgent_customers()
        nearby_customers = self._find_nearby_customers(vehicle_id)
        capacity_constraints = self._assess_capacity_constraints(vehicle_id)
        candidate_customers = list(dict.fromkeys(urgent_customers + nearby_customers))
        recommended_route = self._calculate_optimal_route(vehicle_id, candidate_customers)
        route_distance = self._calculate_route_distance(recommended_route)

        reasoning = [
            f"Vehicle {vehicle_id} has capacity {vehicle.capacity} units.",
            f"{len(urgent_customers)} customers meet the urgent-stock rule.",
            f"{len(nearby_customers)} customers meet the depot-proximity rule.",
            "Customers are ordered by a nearest-neighbor distance heuristic.",
            f"Estimated route distance, including return to the depot: {route_distance:.1f} coordinate units.",
        ]
        alternatives = [
            {
                "route": urgent_customers[:3],
                "reason": "Prioritize customers classified as urgent.",
                "impact": "May omit nearby customers that are not urgent.",
            },
            {
                "route": nearby_customers[:4],
                "reason": "Prioritize customers close to the depot.",
                "impact": "May omit urgent customers outside the proximity set.",
            },
        ]
        risk_assessment = {
            "demand_uncertainty": 0.3,
            "capacity_risk": 0.2,
            "travel_delay_risk": 0.15,
            "stockout_risk": 0.25,
        }
        confidence = self._calculate_confidence(
            urgent_customers, nearby_customers, capacity_constraints
        )

        return AIRecommendation(
            recommendation_type="route",
            action=f"Route vehicle {vehicle_id} through customers {recommended_route}.",
            confidence=confidence,
            reasoning=reasoning,
            alternatives=alternatives,
            expected_impact={},
            risk_assessment=risk_assessment,
        )

    def generate_inventory_recommendation(self, period: int, customer_id: int) -> AIRecommendation:
        """Recommend a replenishment quantity using stock-coverage rules."""
        customer = next(c for c in self.customers if c.id == customer_id)
        current_inventory = self.current_inventory[customer_id]
        days_of_stock = current_inventory / customer.demand_mean
        urgency_level = self._calculate_urgency(customer_id)
        demand_forecast = self._forecast_demand(customer_id)
        
        if days_of_stock < 2:
            target_inventory = customer.max_inventory
            confidence = 0.9
        elif days_of_stock < 4:
            target_inventory = min(customer.max_inventory, 2 * customer.min_inventory)
            confidence = 0.8
        else:
            target_inventory = current_inventory
            confidence = 0.7
        order_quantity = max(0.0, target_inventory - current_inventory)
        action = (
            f"Recommend delivering {order_quantity:.1f} units to customer {customer_id}."
            if order_quantity > 0
            else f"Recommend no immediate delivery to customer {customer_id}."
        )
        reasoning = [
            f"Customer {customer_id} has {current_inventory:.1f} units in the modeled inventory snapshot.",
            f"That inventory covers approximately {days_of_stock:.1f} days at mean demand.",
            f"Mean daily demand is {customer.demand_mean:.1f} units.",
            f"The minimum inventory parameter is {customer.min_inventory:.1f} units.",
            f"The next-period demand forecast is {demand_forecast:.1f} units.",
            f"Heuristic urgency score: {urgency_level:.2f}, where larger values indicate greater urgency.",
        ]
        alternatives = [
            {
                "action": "Choose a larger replenishment quantity.",
                "reason": "A larger quantity may reduce future delivery frequency.",
                "impact": "Higher inventory may increase holding cost.",
            },
            {
                "action": "Choose a smaller replenishment quantity.",
                "reason": "A smaller quantity may limit inventory held at the customer.",
                "impact": "More frequent deliveries may be needed.",
            },
        ]
        risk_assessment = {
            "demand_spike_risk": 0.35,
            "holding_cost_risk": 0.2,
            "stockout_risk": 0.4,
            "forecast_error_risk": 0.25,
        }
        return AIRecommendation(
            recommendation_type="inventory",
            action=action,
            confidence=confidence,
            reasoning=reasoning,
            alternatives=alternatives,
            expected_impact={},
            risk_assessment=risk_assessment,
        )

    def _identify_urgent_customers(self) -> List[int]:
        """Select customers with fewer than two days of mean-demand coverage."""
        return [
            customer.id
            for customer in self.customers
            if self.current_inventory[customer.id] / customer.demand_mean < 2
        ]

    def _find_nearby_customers(self, vehicle_id: int) -> List[int]:
        """Select up to five customers within 30 units of the depot."""
        nearby = []
        for customer in self.customers:
            distance = np.hypot(customer.x - self.depot.x, customer.y - self.depot.y)
            if distance < 30:
                nearby.append(customer.id)
        return nearby[:5]

    def _assess_capacity_constraints(self, vehicle_id: int) -> Dict[str, float]:
        """Report nominal capacity indicators for the current route snapshot."""
        vehicle = next(v for v in self.vehicles if v.id == vehicle_id)
        current_load = sum(
            self.current_inventory[customer_id]
            for customer_id in self.current_routes[vehicle_id]
        )
        utilization = current_load / vehicle.capacity
        return {
            "available_capacity": vehicle.capacity - current_load,
            "utilization_rate": utilization,
            "constraint_level": utilization * 100,
        }

    def _calculate_optimal_route(self, vehicle_id: int, customers: List[int]) -> List[int]:
        """Order candidate stops with a nearest-neighbor heuristic, not an exact TSP solver."""
        unvisited = list(dict.fromkeys(customers))
        route = []
        current_pos = (self.depot.x, self.depot.y)
        customer_by_id = {customer.id: customer for customer in self.customers}
        while unvisited:
            nearest_customer = min(
                unvisited,
                key=lambda customer_id: np.hypot(
                    customer_by_id[customer_id].x - current_pos[0],
                    customer_by_id[customer_id].y - current_pos[1],
                ),
            )
            route.append(nearest_customer)
            unvisited.remove(nearest_customer)
            customer = customer_by_id[nearest_customer]
            current_pos = (customer.x, customer.y)
        return route

    def _calculate_route_distance(self, route: List[int]) -> float:
        """Sum Euclidean leg lengths, including the return leg to the depot."""
        if not route:
            return 0.0
        customer_by_id = {customer.id: customer for customer in self.customers}
        current_pos = (self.depot.x, self.depot.y)
        total_distance = 0.0
        for customer_id in route:
            customer = customer_by_id[customer_id]
            total_distance += np.hypot(customer.x - current_pos[0], customer.y - current_pos[1])
            current_pos = (customer.x, customer.y)
        total_distance += np.hypot(
            current_pos[0] - self.depot.x, current_pos[1] - self.depot.y
        )
        return float(total_distance)

    def _calculate_confidence(
        self, urgent: List[int], nearby: List[int], constraints: Dict[str, float]
    ) -> float:
        """Apply the example's bounded confidence heuristic; this is not calibration."""
        confidence = 0.7 + min(1.0, len(urgent) / 3) * 0.2
        utilization = constraints["utilization_rate"]
        confidence += 0.1 if 0.6 <= utilization <= 0.9 else -0.1
        if len(self.performance_history) > 5:
            confidence += 0.1
        return max(0.5, min(0.95, confidence))

    def _calculate_urgency(self, customer_id: int) -> float:
        """Map stock coverage to a simple urgency score."""
        customer = next(c for c in self.customers if c.id == customer_id)
        days_of_stock = self.current_inventory[customer_id] / customer.demand_mean
        if days_of_stock < 1:
            return 1.0
        if days_of_stock < 2:
            return 0.8
        if days_of_stock < 4:
            return 0.6
        return 0.3

    def _forecast_demand(self, customer_id: int) -> float:
        """Create a reproducible illustrative forecast around mean demand."""
        customer = next(c for c in self.customers if c.id == customer_id)
        variation = self.rng.normal(0, customer.demand_std * 0.3)
        return float(max(0.0, customer.demand_mean + variation))

    def learn_from_feedback(self, feedback: HumanFeedback):
        """Store feedback as a record; accepted patterns do not train a predictive model."""
        self.decision_history.append(feedback)
        if feedback.accepted:
            self.expert_patterns.append(
                {
                    "context": "accepted",
                    "modifications": feedback.modifications,
                    "reasoning": feedback.reasoning,
                }
            )
        self._update_performance_metrics()

    def _update_performance_metrics(self):
        """Track the recent acceptance fraction as a descriptive history."""
        if self.decision_history:
            recent_decisions = self.decision_history[-10:]
            acceptance_rate = sum(d.accepted for d in recent_decisions) / len(recent_decisions)
            self.performance_history.append(acceptance_rate)


class HumanExpert:
    """Simulate expert review using explicit, inspectable rules."""

    def __init__(self, expertise_level: float = 0.8):
        self.expertise_level = expertise_level
        self.decision_patterns = []
        self.preferences = {}
        self.risk_tolerance = 0.3
        self.customer_priorities = {}
        self.route_preferences = {}
        self.inventory_strategies = {}
        self._initialize_expert_knowledge()

    def _initialize_expert_knowledge(self):
        """Set illustrative priorities and preferences for the simulated reviewer."""
        self.customer_priorities = {
            1: 0.9, 2: 0.8, 3: 0.7, 4: 0.6, 5: 0.8,
            6: 0.7, 7: 0.9, 8: 0.6, 9: 0.8, 10: 0.7,
            11: 0.8, 12: 0.7, 13: 0.6, 14: 0.9, 15: 0.8,
        }
        self.route_preferences = {
            "efficiency_weight": 0.6,
            "service_weight": 0.4,
            "max_route_length": 4,
            "prefer_clustering": True,
        }
        self.inventory_strategies = {
            "safety_stock_multiplier": 1.5,
            "reorder_point_multiplier": 1.2,
            "batch_size_preference": "medium",
        }

    def review_recommendation(self, recommendation: AIRecommendation) -> HumanFeedback:
        """Accept or modify a recommendation according to the simulated review rule."""
        expert_confidence = self._evaluate_recommendation(recommendation)
        accepted = expert_confidence > 0.7 and recommendation.confidence > 0.7
        if accepted:
            modifications = {}
            reasoning = "The recommendation meets the simulated expert's review thresholds."
        else:
            modifications = self._generate_modifications(recommendation)
            reasoning = "The simulated expert requests changes based on the stated domain rules."
        decision_confidence = min(
            1.0, expert_confidence + recommendation.confidence * 0.3
        )
        expert_notes = self._generate_expert_notes(
            recommendation, accepted, modifications
        )
        return HumanFeedback(
            recommendation_id=f"REC_{len(self.decision_patterns) + 1}",
            accepted=accepted,
            modifications=modifications,
            reasoning=reasoning,
            confidence_in_decision=decision_confidence,
            expert_notes=expert_notes,
        )

    def _evaluate_recommendation(self, recommendation: AIRecommendation) -> float:
        """Score explanation completeness and the expert's configured expertise."""
        score = 0.5
        if len(recommendation.reasoning) >= 4:
            score += 0.1
        if len(recommendation.alternatives) >= 2:
            score += 0.1
        if recommendation.risk_assessment and max(recommendation.risk_assessment.values()) < 0.5:
            score += 0.1
        return min(1.0, score + self.expertise_level * 0.2)

    def _generate_modifications(self, recommendation: AIRecommendation) -> Dict[str, Any]:
        """Describe the type of change the simulated reviewer would request."""
        if recommendation.recommendation_type == "route":
            return {
                "route_adjustment": "Review priorities for high-value customers.",
                "efficiency_tradeoff": "Consider service needs alongside route length.",
            }
        if recommendation.recommendation_type == "inventory":
            return {
                "safety_stock": "Review the safety-stock assumption.",
                "delivery_frequency": "Review delivery frequency against operating needs.",
            }
        return {}

    def _generate_expert_notes(
        self, recommendation: AIRecommendation, accepted: bool, modifications: Dict[str, Any]
    ) -> str:
        """Summarize the simulated review decision."""
        if accepted:
            return (
                f"Accepted the {recommendation.recommendation_type} recommendation; "
                f"confidence={recommendation.confidence:.2f}."
            )
        return (
            f"Requested changes to the {recommendation.recommendation_type} recommendation; "
            f"modification fields={list(modifications)}."
        )


class HumanAIPartnership:
    """Coordinate recommendations, simulated review, feedback, and metrics."""

    def __init__(self, depot, customers, vehicles, num_periods):
        self.depot = depot
        self.customers = customers
        self.vehicles = vehicles
        self.num_periods = num_periods
        self.ai_system = ExplainableAI(depot, customers, vehicles, num_periods)
        self.human_expert = HumanExpert()
        self.trust_metrics = TrustMetrics(0.5, 0.5, 0.5, 0.5, 0.0, 0.3)
        self.partnership_history = []
        self.performance_timeline = []
        print(
            f"Initialized Human-AI Partnership for {len(customers)} customers "
            f"and {len(vehicles)} vehicles."
        )

    def collaborative_decision_making(self, period: int) -> Dict[str, Any]:
        """Run recommendation, review, feedback, and metric updates for one period."""
        print(f"\n=== COLLABORATIVE DECISION-MAKING: Period {period + 1} ===")
        period_decisions = []
        performance = {
            "ai_recommendations": 0,
            "human_modifications": 0,
            "agreements": 0,
            "disagreements": 0,
            "avg_confidence": 0.0,
            "decision_quality": 0.0,
        }

        print("\nPhase 1: AI recommendation generation")
        recommendations = self._generate_ai_recommendations(period)
        performance["ai_recommendations"] = len(recommendations)

        print("\nPhase 2: Simulated expert review")
        feedback_list = []
        for index, recommendation in enumerate(recommendations, start=1):
            print(f"\n  Recommendation {index}: {recommendation.recommendation_type.upper()}")
            print(f"    AI action: {recommendation.action}")
            print(f"    Heuristic confidence: {recommendation.confidence:.2f}")
            print(f"    Explanation: {recommendation.reasoning[0]}")
            feedback = self.human_expert.review_recommendation(recommendation)
            feedback_list.append(feedback)
            print(f"    Simulated expert decision: {'ACCEPT' if feedback.accepted else 'MODIFY'}")
            print(f"    Decision confidence: {feedback.confidence_in_decision:.2f}")
            if feedback.accepted:
                performance["agreements"] += 1
            else:
                performance["disagreements"] += 1
                performance["human_modifications"] += 1
            self.ai_system.learn_from_feedback(feedback)
            period_decisions.append(
                {
                    "recommendation": recommendation,
                    "feedback": feedback,
                    "final_action": self._implement_decision(recommendation, feedback),
                }
            )

        if feedback_list:
            performance["avg_confidence"] = float(
                np.mean([feedback.confidence_in_decision for feedback in feedback_list])
            )
            performance["decision_quality"] = self._calculate_decision_quality(feedback_list)
        self._update_trust_metrics(performance)
        self.partnership_history.append(
            {"period": period, "decisions": period_decisions, "performance": performance}
        )
        self.performance_timeline.append(performance)

        total_decisions = performance["agreements"] + performance["disagreements"]
        agreement_rate = performance["agreements"] / total_decisions if total_decisions else 0.0
        print(f"\nPeriod {period + 1} summary:")
        print(f"  AI recommendations: {performance['ai_recommendations']}")
        print(f"  Human modifications: {performance['human_modifications']}")
        print(f"  Agreement rate: {agreement_rate:.1%}")
        print(f"  Average decision confidence: {performance['avg_confidence']:.2f}")
        print(f"  Rule-based decision-quality score: {performance['decision_quality']:.2f}")
        print(f"  Overall trust indicator: {self.trust_metrics.overall_trust:.2f}")
        return performance

    def _generate_ai_recommendations(self, period: int) -> List[AIRecommendation]:
        """Generate route and urgent-customer inventory recommendations."""
        recommendations = [
            self.ai_system.generate_route_recommendation(period, vehicle.id)
            for vehicle in self.vehicles[:2]
        ]
        urgent_customers = self.ai_system._identify_urgent_customers()
        recommendations.extend(
            self.ai_system.generate_inventory_recommendation(period, customer_id)
            for customer_id in urgent_customers[:3]
        )
        return recommendations

    def _implement_decision(self, recommendation: AIRecommendation, feedback: HumanFeedback) -> str:
        """Record whether the recommendation was accepted or modified."""
        if feedback.accepted:
            return f"Accepted: {recommendation.action}"
        modifications = "; ".join(
            f"{key}: {value}" for key, value in feedback.modifications.items()
        )
        return f"Modified: {recommendation.action} | {modifications}"

    def _calculate_decision_quality(self, feedback_list: List[HumanFeedback]) -> float:
        """Combine decision confidence and acceptance fraction using stated weights."""
        if not feedback_list:
            return 0.5
        average_confidence = np.mean(
            [feedback.confidence_in_decision for feedback in feedback_list]
        )
        agreement_rate = sum(feedback.accepted for feedback in feedback_list) / len(feedback_list)
        return float(0.7 * average_confidence + 0.3 * agreement_rate)

    def _update_trust_metrics(self, period_performance: Dict[str, float]):
        """Update descriptive indicators with the example's exponential averages."""
        total = period_performance["agreements"] + period_performance["disagreements"]
        current_agreement = (
            period_performance["agreements"] / total if total else 0.5
        )
        self.trust_metrics.agreement_rate = (
            0.7 * self.trust_metrics.agreement_rate + 0.3 * current_agreement
        )
        trust_factor = (current_agreement + period_performance["avg_confidence"]) / 2
        self.trust_metrics.overall_trust = (
            0.8 * self.trust_metrics.overall_trust + 0.2 * trust_factor
        )
        self.trust_metrics.reliability_score = (
            0.8 * self.trust_metrics.reliability_score
            + 0.2 * period_performance["decision_quality"]
        )
        if len(self.performance_timeline) > 1:
            previous_confidence = self.performance_timeline[-1]["avg_confidence"]
            improvement = period_performance["avg_confidence"] - previous_confidence
            self.trust_metrics.learning_progress = float(
                np.clip(self.trust_metrics.learning_progress + improvement * 0.1, 0, 1)
            )
        self.trust_metrics.relationship_strength = (
            0.4 * self.trust_metrics.overall_trust
            + 0.3 * self.trust_metrics.agreement_rate
            + 0.3 * self.trust_metrics.learning_progress
        )

    def run_partnership_simulation(self) -> Dict[str, Any]:
        """Run the configured periods and return aggregate descriptive indicators."""
        print("\n=== HUMAN-AI PARTNERSHIP SIMULATION ===")
        print(f"Simulating {self.num_periods} periods of collaborative decision-making.")
        for period in range(self.num_periods):
            self.collaborative_decision_making(period)
        final_results = self._calculate_final_results()
        print("\n=== SIMULATION COMPLETED ===")
        print(f"Final trust indicator: {self.trust_metrics.overall_trust:.2f}")
        print(f"Final agreement indicator: {self.trust_metrics.agreement_rate:.2f}")
        print(f"Final reliability indicator: {self.trust_metrics.reliability_score:.2f}")
        print(f"Learning-progress indicator: {self.trust_metrics.learning_progress:.2f}")
        print(f"Relationship-strength indicator: {self.trust_metrics.relationship_strength:.2f}")
        return final_results

    def _calculate_final_results(self) -> Dict[str, Any]:
        """Aggregate recommendation counts and period-level indicators."""
        if not self.performance_timeline:
            return {}
        total_recommendations = sum(p["ai_recommendations"] for p in self.performance_timeline)
        total_modifications = sum(p["human_modifications"] for p in self.performance_timeline)
        total_agreements = sum(p["agreements"] for p in self.performance_timeline)
        total_disagreements = sum(p["disagreements"] for p in self.performance_timeline)
        all_decisions = total_agreements + total_disagreements
        return {
            "total_recommendations": total_recommendations,
            "total_modifications": total_modifications,
            "total_agreements": total_agreements,
            "total_disagreements": total_disagreements,
            "overall_agreement_rate": total_agreements / all_decisions if all_decisions else 0.0,
            "avg_confidence": float(np.mean([p["avg_confidence"] for p in self.performance_timeline])),
            "avg_decision_quality": float(np.mean([p["decision_quality"] for p in self.performance_timeline])),
            "final_trust_metrics": self.trust_metrics,
            "learning_trajectory": [p["avg_confidence"] for p in self.performance_timeline],
        }

In [ ]:
def create_partnership_instance():
    """Create the illustrative depot, customer, and vehicle data."""
    depot = type(
        "Depot",
        (),
        {"x": 0, "y": 0, "initial_inventory": 1200, "holding_cost": 0.1},
    )()
    np.random.seed(42)
    customers = []
    for index in range(15):
        cluster_center_x = np.random.choice([-20, 0, 20])
        cluster_center_y = np.random.choice([-20, 0, 20])
        x = cluster_center_x + np.random.normal(0, 8)
        y = cluster_center_y + np.random.normal(0, 8)
        customer = type(
            "Customer",
            (),
            {
                "id": index + 1,
                "x": x,
                "y": y,
                "demand_mean": np.random.uniform(10, 25),
                "demand_std": np.random.uniform(2, 5),
                "holding_cost": np.random.uniform(0.1, 0.3),
                "initial_inventory": np.random.uniform(40, 80),
                "min_inventory": np.random.uniform(10, 20),
                "max_inventory": np.random.uniform(90, 130),
            },
        )()
        customers.append(customer)
    vehicles = [
        type(
            "Vehicle",
            (),
            {"id": vehicle_id, "capacity": 100, "fixed_cost": 120, "variable_cost": 1.8},
        )()
        for vehicle_id in range(1, 5)
    ]
    return depot, customers, vehicles


print("Creating the Human-AI Partnership example instance.")
depot, customers, vehicles = create_partnership_instance()
print(f"Depot: ({depot.x}, {depot.y}); initial inventory: {depot.initial_inventory} units")
print("Customers (first five shown):")
for customer in customers[:5]:
    print(
        f"  C{customer.id}: ({customer.x:.1f}, {customer.y:.1f}); "
        f"mean demand {customer.demand_mean:.1f} +/- {customer.demand_std:.1f}; "
        f"initial inventory {customer.initial_inventory:.1f}"
    )
print("Vehicles:")
for vehicle in vehicles:
    print(
        f"  V{vehicle.id}: capacity {vehicle.capacity} units; "
        f"fixed cost {vehicle.fixed_cost}"
    )

In [ ]:
def run_partnership_simulation(depot, customers, vehicles, num_periods=8):
    """Create and run the partnership simulation for the supplied instance."""
    partnership = HumanAIPartnership(depot, customers, vehicles, num_periods)
    print("\n=== HUMAN-AI PARTNERSHIP SYSTEM ===")
    print("AI component: explainable, rule-based recommendations")
    print(f"Simulated expert expertise parameter: {partnership.human_expert.expertise_level:.1f}")
    print(f"Initial trust indicator: {partnership.trust_metrics.overall_trust:.2f}")
    results = partnership.run_partnership_simulation()
    return partnership, results


partnership, results = run_partnership_simulation(depot, customers, vehicles, 8)

In [ ]:
def analyze_partnership_performance(partnership, results):
    """Report the simulation's descriptive indicators and their interpretation."""
    print("\n=== HUMAN-AI PARTNERSHIP ANALYSIS ===")
    trust_metrics = results["final_trust_metrics"]
    print("\nTrust and relationship indicators (rule-defined, not empirical measures):")
    print(f"  Overall trust indicator: {trust_metrics.overall_trust:.2f} (initial value: 0.50)")
    print(f"  Smoothed agreement indicator: {trust_metrics.agreement_rate:.2f}")
    print(f"  Reliability indicator: {trust_metrics.reliability_score:.2f}")
    print(f"  Learning-progress indicator: {trust_metrics.learning_progress:.2f}")
    print(f"  Relationship-strength indicator: {trust_metrics.relationship_strength:.2f}")

    print("\nDecision-process summary:")
    print(f"  Recommendations: {results['total_recommendations']}")
    print(f"  Simulated modifications: {results['total_modifications']}")
    print(f"  Agreements: {results['total_agreements']}")
    print(f"  Disagreements: {results['total_disagreements']}")
    print(f"  Overall agreement fraction: {results['overall_agreement_rate']:.1%}")
    print(f"  Mean decision confidence: {results['avg_confidence']:.2f}")
    print(f"  Mean rule-based decision-quality score: {results['avg_decision_quality']:.2f}")

    trajectory = results["learning_trajectory"]
    if trajectory:
        print("\nConfidence trajectory:")
        print(f"  First-period mean: {trajectory[0]:.2f}")
        print(f"  Last-period mean: {trajectory[-1]:.2f}")
        print(
            "  Change across the observed periods: "
            f"{trajectory[-1] - trajectory[0]:+.2f}"
        )
        print("  This is a change in the confidence indicator, not measured model learning.")

    expert_patterns = partnership.ai_system.expert_patterns
    print("\nFeedback records:")
    print(f"  Accepted patterns stored: {len(expert_patterns)}")
    print(f"  Decisions recorded: {len(partnership.ai_system.decision_history)}")
    print(f"  Acceptance-history observations: {len(partnership.ai_system.performance_history)}")
    print("  Stored patterns are records; they are not used to retrain the recommendation rules.")

    collaboration_efficiency = results["overall_agreement_rate"] * results["avg_confidence"]
    knowledge_transfer = trust_metrics.learning_progress * trust_metrics.relationship_strength
    decision_improvement = trust_metrics.learning_progress * 100
    print("\nComposite indicators:")
    print(f"  Agreement-confidence product: {collaboration_efficiency:.2f}")
    print(f"  Learning-progress/relationship product: {knowledge_transfer:.2f}")
    print(f"  Learning-progress indicator expressed as a percentage: {decision_improvement:.1f}%")
    return {
        "trust_development": trust_metrics.overall_trust - 0.5,
        "agreement_rate": results["overall_agreement_rate"],
        "learning_improvement": trust_metrics.learning_progress,
        "collaboration_efficiency": collaboration_efficiency,
        "knowledge_transfer": knowledge_transfer,
        "decision_improvement": decision_improvement,
    }


performance_analysis = analyze_partnership_performance(partnership, results)

In [ ]:
def visualize_partnership_results(partnership, results, performance_analysis):
    """Plot the partnership indicators and the generated instance locations."""
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    fig.suptitle("Human-AI Partnership: Descriptive Simulation Results", fontsize=16, fontweight="bold")

    ax = axes[0, 0]
    periods = np.arange(1, len(partnership.performance_timeline) + 1)
    confidence = [p["avg_confidence"] for p in partnership.performance_timeline]
    agreement = [
        p["agreements"] / (p["agreements"] + p["disagreements"])
        if p["agreements"] + p["disagreements"]
        else 0.5
        for p in partnership.performance_timeline
    ]
    ax.plot(periods, confidence, marker="o", linewidth=2, label="Decision confidence")
    ax.plot(periods, agreement, marker="s", linewidth=2, label="Agreement fraction")
    ax.set(title="Confidence and Agreement by Period", xlabel="Period", ylabel="Indicator (0-1)")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[0, 1]
    metric_names = ["Agreement", "Confidence", "Decision quality", "Trust"]
    metric_values = [
        results["overall_agreement_rate"],
        results["avg_confidence"],
        results["avg_decision_quality"],
        results["final_trust_metrics"].overall_trust,
    ]
    bars = ax.bar(metric_names, metric_values, color=["green", "blue", "purple", "orange"], alpha=0.75)
    ax.set(title="Summary Indicators", ylabel="Value (0-1)", ylim=(0, 1))
    ax.tick_params(axis="x", rotation=35)
    ax.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, metric_values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{value:.2f}", ha="center")

    ax = axes[0, 2]
    labels = ["Agreements", "Modifications", "Recommendations"]
    counts = [results["total_agreements"], results["total_modifications"], results["total_recommendations"]]
    bars = ax.bar(labels, counts, color=["green", "orange", "blue"], alpha=0.75)
    ax.set(title="Decision-Process Counts", ylabel="Count")
    ax.tick_params(axis="x", rotation=35)
    ax.grid(True, axis="y", alpha=0.3)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(count), ha="center", va="bottom")

    ax = axes[1, 0]
    trajectory = results["learning_trajectory"]
    trajectory_periods = np.arange(1, len(trajectory) + 1)
    ax.plot(trajectory_periods, trajectory, color="purple", marker="o", linewidth=2)
    ax.fill_between(trajectory_periods, trajectory, alpha=0.25, color="purple")
    ax.set(title="Mean Confidence by Period", xlabel="Period", ylabel="Heuristic confidence (0-1)", ylim=(0, 1))
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    trust = results["final_trust_metrics"]
    categories = ["Trust", "Agreement", "Reliability", "Learning progress", "Relationship"]
    values = [trust.overall_trust, trust.agreement_rate, trust.reliability_score,
              trust.learning_progress, trust.relationship_strength]
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False)
    closed_values = values + values[:1]
    closed_angles = np.append(angles, angles[0])
    ax.plot(closed_angles, closed_values, "o-", linewidth=2, color="blue", label="Final indicators")
    ax.fill(closed_angles, closed_values, alpha=0.2, color="blue")
    ax.set_xticks(angles)
    ax.set_xticklabels(categories, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_title("Final Indicator Profile")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))

    ax = axes[1, 2]
    patterns = partnership.ai_system.expert_patterns
    if patterns:
        pattern_types = sorted({pattern["context"] for pattern in patterns})
        pattern_counts = [sum(pattern["context"] == kind for pattern in patterns) for kind in pattern_types]
        bars = ax.bar(pattern_types, pattern_counts, color="teal", alpha=0.75)
        for bar, count in zip(bars, pattern_counts):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(count), ha="center", va="bottom")
    else:
        ax.text(0.5, 0.5, "No accepted patterns recorded", ha="center", va="center", transform=ax.transAxes)
    ax.set(title="Stored Accepted Feedback Patterns", ylabel="Count")
    ax.grid(True, axis="y", alpha=0.3)

    ax = axes[2, 0]
    efficiency_data = np.array(
        [
            [
                period["avg_confidence"],
                period["agreements"] / (period["agreements"] + period["disagreements"])
                if period["agreements"] + period["disagreements"]
                else 0.5,
                period["decision_quality"],
            ]
            for period in partnership.performance_timeline
        ]
    ).T
    image = ax.imshow(efficiency_data, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    ax.set(title="Period-Level Indicators", xlabel="Period", ylabel="Indicator")
    ax.set_xticks(np.arange(len(periods)))
    ax.set_xticklabels(periods)
    ax.set_yticks([0, 1, 2])
    ax.set_yticklabels(["Confidence", "Agreement", "Decision quality"])
    colorbar = fig.colorbar(image, ax=ax)
    colorbar.set_label("Indicator value (0-1)")

    ax = axes[2, 1]
    qualities = [p["decision_quality"] for p in partnership.performance_timeline]
    ax.plot(periods, qualities, color="green", marker="o", linewidth=2)
    ax.fill_between(periods, qualities, alpha=0.2, color="green")
    ax.set(title="Rule-Based Decision-Quality Score", xlabel="Period", ylabel="Score (0-1)", ylim=(0, 1))
    ax.grid(True, alpha=0.3)

    ax = axes[2, 2]
    contributions = [trust.overall_trust * 0.4, trust.agreement_rate * 0.3,
                     trust.learning_progress * 0.3]
    bars = ax.bar(["Trust", "Agreement", "Learning progress"], contributions,
                  color=["blue", "green", "purple"], alpha=0.75)
    for bar, value in zip(bars, contributions):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{value:.2f}", ha="center", va="bottom")
    ax.axhline(trust.relationship_strength, color="red", linestyle="--",
               label=f"Weighted total: {trust.relationship_strength:.2f}")
    ax.set(title="Relationship-Strength Components", ylabel="Weighted contribution")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()

    plt.tight_layout()
    plt.show()


visualize_partnership_results(partnership, results, performance_analysis)

In [ ]:
def compare_partnership_with_approaches(partnership, results):
    """Compare simulated partnership indicators with explicitly illustrative references."""
    approaches = ["Pure AI", "Human Expert", "Human-AI Partnership"]
    decision_quality = {
        "Pure AI": 0.75,
        "Human Expert": 0.85,
        "Human-AI Partnership": results["avg_decision_quality"],
    }
    trust_acceptance = {
        "Pure AI": 0.60,
        "Human Expert": 0.90,
        "Human-AI Partnership": results["final_trust_metrics"].overall_trust,
    }
    learning_adaptation = {
        "Pure AI": 0.70,
        "Human Expert": 0.40,
        "Human-AI Partnership": results["final_trust_metrics"].learning_progress,
    }
    scalability = {
        "Pure AI": 0.90,
        "Human Expert": 0.30,
        "Human-AI Partnership": 0.70,
    }
    print("\n=== PARTNERSHIP AND ILLUSTRATIVE REFERENCE SCORES ===")
    print("Reference values for Pure AI and Human Expert are assumptions, not measured results.")
    print("The partnership column is calculated from this simulation's rule-defined indicators.")
    print(f"\n{'Approach':<22} {'Decision quality':>17} {'Trust':>10} {'Learning':>10} {'Scalability':>13}")
    print("-" * 76)
    for approach in approaches:
        print(
            f"{approach:<22} {decision_quality[approach]:>17.2f} "
            f"{trust_acceptance[approach]:>10.2f} "
            f"{learning_adaptation[approach]:>10.2f} "
            f"{scalability[approach]:>13.2f}"
        )
    print("\nInterpret these values as a discussion aid, not as evidence of comparative performance.")

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Partnership Indicators and Illustrative Reference Scores", fontsize=15, fontweight="bold")
    comparisons = [
        (decision_quality, "Decision quality", "Score (0-1)"),
        (trust_acceptance, "Trust indicator", "Score (0-1)"),
        (learning_adaptation, "Learning/adaptation indicator", "Score (0-1)"),
        (scalability, "Scalability reference", "Illustrative score (0-1)"),
    ]
    colors = ["blue", "green", "purple"]
    for ax, (metric, title, ylabel) in zip(axes.flat, comparisons):
        values = [metric[approach] for approach in approaches]
        bars = ax.bar(approaches, values, color=colors, alpha=0.75)
        ax.set(title=title, ylabel=ylabel, ylim=(0, 1))
        ax.tick_params(axis="x", rotation=25)
        ax.grid(True, axis="y", alpha=0.3)
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                    f"{value:.2f}", ha="center", va="bottom")
    plt.tight_layout()
    plt.show()
    return {
        "decision_quality_difference_vs_reference": (
            decision_quality["Human-AI Partnership"] - decision_quality["Pure AI"]
        ),
        "trust_difference_vs_reference": (
            trust_acceptance["Human-AI Partnership"] - trust_acceptance["Pure AI"]
        ),
        "learning_difference_vs_reference": (
            learning_adaptation["Human-AI Partnership"] - learning_adaptation["Pure AI"]
        ),
        "scalability_difference_vs_reference": (
            scalability["Pure AI"] - scalability["Human-AI Partnership"]
        ),
    }


comparison_results = compare_partnership_with_approaches(partnership, results)

## Key Insights from the Tier 7 Analysis

### What the simulation represents

The example makes a collaborative decision sequence explicit: recommendations include reasoning and alternatives; a simulated expert applies review rules; feedback is recorded; and summary indicators are updated. The route recommendation orders candidate customers with a nearest-neighbor heuristic. The inventory recommendation uses days of stock and target-inventory rules. Neither recommendation is optimized jointly with the other, and neither is executed as an operational plan.

### How to read the indicators

- **Agreement rate** is the fraction of simulated reviews that accept recommendations. It measures agreement under the coded review rule, not correctness.
- **Confidence** is a heuristic score based on the coded conditions. It has not been calibrated against observed outcomes.
- **Decision quality** is defined in the code as $0.7$ times mean decision confidence plus $0.3$ times the acceptance fraction. It is a constructed score, not an independently validated quality measure.
- **Overall trust** is updated from the period's agreement fraction and mean decision confidence. It is a model indicator, not a direct measurement of human trust.
- **Learning progress** tracks changes in mean decision confidence through the coded update rule. Since feedback does not train a predictive model, this should not be interpreted as demonstrated AI learning.
- **Relationship strength** combines the trust, agreement, and learning-progress indicators with weights $0.4$, $0.3$, and $0.3$, respectively.

For a period with at least one reviewed recommendation, let $a_t$ denote its acceptance fraction, $c_t$ its mean decision confidence, and $q_t$ its rule-based decision-quality score. The implemented updates are:

$$\begin{aligned}
A_t &= 0.7A_{t-1} + 0.3a_t, \\
T_t &= 0.8T_{t-1} + 0.2\frac{a_t+c_t}{2}, \\
R_t &= 0.8R_{t-1} + 0.2q_t, \\
L_t &= \max\left(0, L_{t-1} + 0.1(c_t-c_{t-1})\right), \\
S_t &= 0.4T_t + 0.3A_t + 0.3L_t.
\end{aligned}$$

Here, $A_t$ is the smoothed agreement indicator, $T_t$ is the overall trust indicator, $R_t$ is the reliability indicator, $L_t$ is the learning-progress indicator, and $S_t$ is relationship strength. The initial values are set in `HumanAIPartnership.__init__`. These equations describe the notebook's chosen bookkeeping rules; they do not establish that the indicators are valid measures of real organizational outcomes.

### Scope and limitations

The generated data and review rules create a reproducible teaching example, not a controlled comparison of AI, human, and partnership performance. Reference scores for Pure AI and Human Expert in the comparison cell are illustrative assumptions. Customer inventory is held constant across periods; vehicle routes do not enforce load feasibility; and demand uncertainty is not propagated through an inventory balance. The code also does not model human availability, interface design, or actual expert learning.

### Further study

Within this collaborative-decision framework, a fuller study could add operational state updates, validate confidence against observed outcomes, collect real expert feedback, and evaluate recommendation quality against explicit service and cost measures. Such extensions require additional data and assumptions; the indicators in this notebook alone cannot establish their effects.